In [1]:
# STREAMING_CHUNK:Installing standard dependencies...
# Installing sentence-transformers for local embedding generation and pypdf for custom document parsing
!pip install -q sentence-transformers pypdf requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 22.2 MB/s eta 0:00:00


In [2]:
# STREAMING_CHUNK:Defining document ingestion and text splitting logic...
import os
from pypdf import PdfReader

# Custom paragraph corpus that serves as our private document database if no PDF is uploaded.
built_in_document = """
The Retrieval-Augmented Generation (RAG) architecture is a technique in Natural Language Processing.
RAG combines the power of pre-trained language models with external information retrieval systems.
Instead of relying solely on the parametric memory of an LLM, RAG retrieves relevant facts from a custom document database.
The core pipeline consists of three sequential steps: Document Ingestion, Information Retrieval, and Answer Generation.
During ingestion, unstructured documents like PDFs, text files, or markdown articles are parsed into raw strings.
These raw strings are split into smaller units called text chunks to preserve local context and prevent overflow limits.
The optimal chunk size typically ranges from 100 to 500 words, often with an overlap of 10% to 20% to avoid splitting sentences.
An embedding model converts each chunk into a high-dimensional vector representing its semantic meaning.
When a user asks a question, the query is embedded, and a similarity search (like cosine distance) finds the top context matches.
The LLM then receives both the query and the retrieved contexts as an augmented prompt to generate a highly grounded, factual answer.
"""

def extract_text_from_pdf(pdf_path):
    """Parses a local PDF document and extracts clean text string."""
    reader = PdfReader(pdf_path)
    extracted_text = ""
    for page in reader.pages:
        text = page.extract_text()
        if text:
            extracted_text += text + "\n"
    return extracted_text

def chunk_text(text, chunk_size=300, overlap=50):
    """Splits a long document string into overlapping character chunks to maintain structural context."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end].strip())
        start += (chunk_size - overlap)
    # Filter out empty or extremely small fragments
    return [c for c in chunks if len(c) > 10]

# Setup standard testing document
pdf_upload_path = "document.pdf"  # To test your own PDF, upload it to Colab and rename it to document.pdf
if os.path.exists(pdf_upload_path):
    print(f"Detected custom PDF file: {pdf_upload_path}. Parsing content...")
    raw_document_content = extract_text_from_pdf(pdf_upload_path)
else:
    print("No custom PDF detected. Falling back to the built-in system architecture document...")
    raw_document_content = built_in_document

# Generate the processed text chunks
document_chunks = chunk_text(raw_document_content, chunk_size=300, overlap=50)
print(f"Ingested document split into {len(document_chunks)} distinct overlapping chunks.")

No custom PDF detected. Falling back to the built-in system architecture document...
Ingested document split into 5 distinct overlapping chunks.


In [3]:
# STREAMING_CHUNK:Generating sentence transformer vector representations...
from sentence_transformers import SentenceTransformer

# Load a lightweight, state-of-the-art semantic model for local vector creation
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate vector representations for each document chunk
print("Generating high-dimensional embeddings for your custom chunks...")
chunk_embeddings = embedding_model.encode(document_chunks, show_progress_bar=True)
print(f"Successfully indexed vector space with matrix shape: {chunk_embeddings.shape}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating high-dimensional embeddings for your custom chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Successfully indexed vector space with matrix shape: (5, 384)


In [4]:
# STREAMING_CHUNK:Implementing cosine similarity mathematics...
import numpy as np

def cosine_similarity(v1, v2):
    """Calculates semantic alignment score between two vectors."""
    dot_product = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    if norm_v1 == 0 or norm_v2 == 0:
        return 0.0
    return dot_product / (norm_v1 * norm_v2)

def retrieve_relevant_contexts(query, chunks, embeddings, k=2):
    """Finds the top k matching chunks based on semantic similarity."""
    query_vector = embedding_model.encode(query)
    similarity_scores = []

    for idx, chunk_vector in enumerate(embeddings):
        score = cosine_similarity(query_vector, chunk_vector)
        similarity_scores.append((score, chunks[idx]))

    # Sort descending by similarity score
    similarity_scores.sort(key=lambda x: x[0], reverse=True)
    return similarity_scores[:k]

# Quick diagnostic search test
test_query = "What is the optimal size of a chunk?"
retrieved_results = retrieve_relevant_contexts(test_query, document_chunks, chunk_embeddings, k=2)

print(f"\n--- Diagnostic Retrieval Test ---")
print(f"Query: '{test_query}'")
for idx, (score, chunk) in enumerate(retrieved_results):
    print(f"Match {idx+1} [Confidence Score: {score:.4f}]: \"{chunk}\"")


--- Diagnostic Retrieval Test ---
Query: 'What is the optimal size of a chunk?'
Match 1 [Confidence Score: 0.4936]: "t files, or markdown articles are parsed into raw strings.
These raw strings are split into smaller units called text chunks to preserve local context and prevent overflow limits.
The optimal chunk size typically ranges from 100 to 500 words, often with an overlap of 10% to 20% to avoid splitting se"
Match 2 [Confidence Score: 0.2174]: "ith an overlap of 10% to 20% to avoid splitting sentences.
An embedding model converts each chunk into a high-dimensional vector representing its semantic meaning.
When a user asks a question, the query is embedded, and a similarity search (like cosine distance) finds the top context matches.
The LL"


In [5]:
# STREAMING_CHUNK:Configuring API connection endpoints and prompt formatting...
import json
import requests
import time

def call_gemini_llm(prompt, system_instruction=""):
    """Queries the Gemini-3-Flash model directly using standard fetch protocols with exponential backoff."""
    # Leave the API key as empty. Canvas supplies this dynamically in the runtime container.
    api_key = ""
    api_url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-3-flash-preview:generateContent?key={api_key}"

    payload = {
        "contents": [{"parts": [{"text": prompt}]}],
        "systemInstruction": {"parts": [{"text": system_instruction}]}
    }

    # Simple retry block with exponential backoff to handle network throttling
    for delay in [1, 2, 4]:
        try:
            response = requests.post(api_url, headers={"Content-Type": "application/json"}, json=payload)
            if response.status_code == 200:
                result = response.json()
                return result["candidates"][0]["content"]["parts"][0]["text"]
            elif response.status_code == 429:
                time.sleep(delay)
            else:
                return f"LLM API Error (Status {response.status_code}): {response.text}"
        except Exception as e:
            time.sleep(delay)

    return "Failed to fetch response after multiple retries."

def generate_rag_answer(query, retrieved_contexts):
    """Formulates a query augmented with matching context to ground the final generated answer."""
    # Build prompt template injecting retrieved resources
    context_str = "\n\n".join([f"[Source Chunk]: {chunk}" for _, chunk in retrieved_contexts])

    system_rules = (
        "You are a factual, concise documentation assistant. You must answer questions "
        "using ONLY the provided custom source chunks. If the answer cannot be found in the "
        "sources, politely explain that you do not have that information."
    )

    prompt = (
        f"Context information from custom documents:\n"
        f"---------------------\n"
        f"{context_str}\n"
        f"---------------------\n"
        f"Given the context above, answer this question clearly and step-by-step:\n"
        f"Question: {query}\n"
        f"Answer:"
    )

    return call_gemini_llm(prompt, system_instruction=system_rules)

print("Generation pipeline connected and initialized successfully!")

Generation pipeline connected and initialized successfully!


In [6]:
# STREAMING_CHUNK:Testing queries through the end-to-end evaluation pipeline...
# Define questions to evaluate the RAG pipeline's capabilities
eval_questions = [
    "What are the three sequential steps of a RAG pipeline?",
    "Why does the system split document texts into smaller units?",
    "What is the typical optimal chunk size recommended in the system document?"
]

print("==========================================================")
print("            RAG SYSTEM EXECUTION COMPARISON               ")
print("==========================================================\n")

for query in eval_questions:
    print(f"User Query: '{query}'")

    # 1. Retrieve
    matches = retrieve_relevant_contexts(query, document_chunks, chunk_embeddings, k=2)

    # 2. Augment & Generate
    grounded_answer = generate_rag_answer(query, matches)

    print("\n--- Retrieved Grounding Sources ---")
    for idx, (score, chunk) in enumerate(matches):
        print(f"Source {idx+1} [Similarity Score: {score:.4f}]: \"{chunk}\"")

    print("\n--- Model Generated Answer (Grounded in Source) ---")
    print(grounded_answer)
    print("\n" + "="*58 + "\n")


            RAG SYSTEM EXECUTION COMPARISON               

User Query: 'What are the three sequential steps of a RAG pipeline?'

--- Retrieved Grounding Sources ---
Source 1 [Similarity Score: 0.5655]: "y of an LLM, RAG retrieves relevant facts from a custom document database.
The core pipeline consists of three sequential steps: Document Ingestion, Information Retrieval, and Answer Generation.
During ingestion, unstructured documents like PDFs, text files, or markdown articles are parsed into raw"
Source 2 [Similarity Score: 0.2814]: "The Retrieval-Augmented Generation (RAG) architecture is a technique in Natural Language Processing.
RAG combines the power of pre-trained language models with external information retrieval systems.
Instead of relying solely on the parametric memory of an LLM, RAG retrieves relevant facts from a c"

--- Model Generated Answer (Grounded in Source) ---
LLM API Error (Status 403): {
  "error": {
    "code": 403,
    "message": "Method doesn't allow unregi